<a href="https://colab.research.google.com/github/YuYing33/CS521_FA26_HW/blob/main/P3b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P3(b) — OLMo 2 1B post-training evaluation

Four checkpoints × fixed 200 GSM8K + 200 IFEval examples. Deterministic greedy decoding. GSM8K numeric exact match; **official Google IFEval strict prompt-level** checker. Outputs are checkpointed to Drive if mounted.

**Run sequentially.** On a T4, evaluation (especially long IFEval responses) may take considerable time. Set `N=3` for a smoke test first, then reset to `N=200` for the submitted results. Re-running a completed checkpoint will reuse saved outputs. The Base model uses a plain text prompt; chat checkpoints use their own templates.

In [7]:
%pip -q install -U "transformers>=4.48" accelerate datasets sentencepiece absl-py langdetect nltk immutabledict pandas tqdm

In [2]:
%pip install -q pandas==2.2.3

In [6]:
import os
import sys
import subprocess
import importlib.util

repo = "/content/google-research"
module_path = os.path.join(repo, "instruction_following_eval")

# Check whether the IFEval source code is present
if not os.path.isfile(os.path.join(module_path, "evaluation_lib.py")):
    print("Downloading IFEval...")

    if os.path.exists(repo):
        subprocess.run(["rm", "-rf", repo], check=True)

    subprocess.run([
        "git", "clone",
        "--depth", "1",
        "--filter=blob:none",
        "--sparse",
        "https://github.com/google-research/google-research.git",
        repo
    ], check=True)

    subprocess.run([
        "git", "-C", repo,
        "sparse-checkout", "set",
        "instruction_following_eval"
    ], check=True)

sys.path.insert(0, repo)
importlib.invalidate_caches()

from instruction_following_eval import evaluation_lib

print("IFEval imported successfully!")
print("Source:", evaluation_lib.__file__)

IFEval imported successfully!
Source: /content/google-research/instruction_following_eval/evaluation_lib.py


In [8]:
import sys
sys.path.insert(0, "/content/google-research")

from instruction_following_eval import evaluation_lib

print("IFEval imported successfully!")

IFEval imported successfully!


In [9]:
import os, sys, json, re, gc, random, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Select T4 GPU in Runtime settings"
print(torch.cuda.get_device_name(0))
sys.path.insert(0, '/content/google-research')
from instruction_following_eval import evaluation_lib

# Drive prevents loss of generated answers on runtime reset.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/P3b_OLMo_results')
except Exception as exc:
    print('Drive unavailable; using ephemeral /content:', exc)
    OUT = Path('/content/P3b_OLMo_results')
OUT.mkdir(parents=True, exist_ok=True)
print('Results folder:', OUT)

MODEL_IDS = {
    'Base': 'allenai/OLMo-2-0425-1B',
    'SFT': 'allenai/OLMo-2-0425-1B-SFT',
    'DPO': 'allenai/OLMo-2-0425-1B-DPO',
    'RLVR': 'allenai/OLMo-2-0425-1B-Instruct',
}
N = 200  # Use 3 only as a smoke test; switch back to 200 for report
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Same selected examples for every checkpoint, persisted to disk.
gsm_path = OUT / f'gsm8k_indices_{N}_seed{SEED}.json'
ife_path = OUT / f'ifeval_keys_{N}_seed{SEED}.json'
gsm_ds = load_dataset('openai/gsm8k', 'main', split='test')
ife_ds = load_dataset('google/IFEval', split='train')
if gsm_path.exists():
    gsm_indices = json.loads(gsm_path.read_text())
else:
    gsm_indices = sorted(random.Random(SEED).sample(range(len(gsm_ds)), N))
    gsm_path.write_text(json.dumps(gsm_indices))
if ife_path.exists():
    ife_keys = json.loads(ife_path.read_text())
else:
    ife_keys = sorted(random.Random(SEED).sample([int(r['key']) for r in ife_ds], N))
    ife_path.write_text(json.dumps(ife_keys))
gsm_examples = [(int(i), gsm_ds[int(i)]) for i in gsm_indices]
ife_by_key = {int(row['key']):row for row in ife_ds}
ife_examples = [(int(k), ife_by_key[int(k)]) for k in ife_keys]
print('GSM8K',len(gsm_examples),'IFEval',len(ife_examples))

Tesla T4
Mounted at /content/drive
Results folder: /content/drive/MyDrive/P3b_OLMo_results


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

README.md:   0%|          | 0.00/5.52k [00:00<?, ?B/s]

ifeval_input_data.jsonl:   0%|          | 0.00/207k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

GSM8K 200 IFEval 200


In [10]:
# A consistent task prompt is applied to every model. Chat-stage models
# receive the same user content in their native template.
def make_prompt(tok, stage, task, question):
    if task == 'gsm8k':
        text = ("Solve the math problem. Show your reasoning briefly. "
                "Finish with a line formatted exactly as 'Final answer: NUMBER'.\n\n"
                + question)
    else:
        text = question
    if stage == 'Base':
        return 'Question:\n' + text + '\nAnswer:\n'
    return tok.apply_chat_template(
        [{'role':'user','content':text}],
        tokenize=False, add_generation_prompt=True
    )

# Numeric exact match, not substring match; unparseable outputs count as incorrect.
NUMBER = r'[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?'
def normalize_number(value):
    if value is None: return None
    try:
        from decimal import Decimal
        return str(Decimal(value.replace(',','').strip()).normalize())
    except Exception:
        return None

def extract_answer(text):
    # Explicit answer tag first, then boxed response, then terminal number.
    tags = [
        rf'Final answer\s*:\s*\$?({NUMBER})',
        rf'\\boxed\{{\s*({NUMBER})\s*\}}',
        rf'(?i:the answer is)\s*\$?({NUMBER})'
    ]
    for pattern in tags:
        found = re.findall(pattern, text, flags=re.I)
        if found: return normalize_number(found[-1])
    found = re.findall(NUMBER, text)
    return normalize_number(found[-1]) if found else None

def generate_one(model, tok, prompt, tokens):
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(output[0,inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

def jsonl_read(path):
    if not path.exists(): return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
def append_record(path, rec):
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(rec,ensure_ascii=False)+'\n')

def evaluate_checkpoint(stage):
    mid=MODEL_IDS[stage]
    print('Loading', stage, mid)
    tok=AutoTokenizer.from_pretrained(mid)
    model=AutoModelForCausalLM.from_pretrained(mid, torch_dtype=torch.float16, device_map='auto').eval()
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    try:
        for task, examples, cap in [('gsm8k',gsm_examples,384),('ifeval',ife_examples,768)]:
            path=OUT/f'{stage}_{task}_{N}.jsonl'
            old=jsonl_read(path)
            completed={int(r['id']) for r in old}
            todo=[(i,r) for i,r in examples if i not in completed]
            print(stage, task, len(completed), 'already finished,', len(todo), 'remaining')
            for idx, row in tqdm(todo, desc=f'{stage} {task}'):
                question=row['question'] if task=='gsm8k' else row['prompt']
                prompt=make_prompt(tok,stage,task,question)
                answer=generate_one(model,tok,prompt,cap)
                rec={'id':idx,'prompt':question,'response':answer,
                     'stage':stage,'task':task}
                if task=='gsm8k':
                    rec['gold']=normalize_number(row['answer'].split('####')[-1])
                    rec['prediction']=extract_answer(answer)
                    rec['correct']=(rec['prediction'] is not None and rec['prediction']==rec['gold'])
                append_record(path, rec) # saves after EVERY sample
    finally:
        del model, tok
        gc.collect(); torch.cuda.empty_cache()
    print('Completed:',stage)

# Run one at a time; completed generations will be skipped.
for stage in MODEL_IDS:
    evaluate_checkpoint(stage)

Loading Base allenai/OLMo-2-0425-1B


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (49 outputs left out)

RLVR gsm8k:   0%|          | 0/200 [00:00<?, ?it/s]

RLVR ifeval 0 already finished, 200 remaining


RLVR ifeval:   0%|          | 0/200 [00:00<?, ?it/s]

Completed: RLVR


In [15]:

from instruction_following_eval import instructions_registry

def clean_ifeval_kwargs(row):
    cleaned = []

    for instruction_id, raw_kwargs in zip(
        row["instruction_id_list"],
        row["kwargs"]
    ):
        instruction_cls = (
            instructions_registry.INSTRUCTION_DICT[instruction_id]
        )
        instruction = instruction_cls(instruction_id)

        allowed_keys = set(
            instruction.get_instruction_args_keys() or []
        )

        filtered_kwargs = {
            key: value
            for key, value in (raw_kwargs or {}).items()
            if key in allowed_keys
        }

        cleaned.append(filtered_kwargs)

    return cleaned

print("clean_ifeval_kwargs is ready!")


clean_ifeval_kwargs is ready!


In [17]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

print("NLTK resources ready!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK resources ready!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [18]:
# Score with the actual Google IFEval checker (strict and loose).
# IFEval records contain ALL required metadata, including kwargs.
summary=[]
for stage in MODEL_IDS:
    gsm_path=OUT/f'{stage}_gsm8k_{N}.jsonl'
    ife_path=OUT/f'{stage}_ifeval_{N}.jsonl'
    gsm=jsonl_read(gsm_path); ife=jsonl_read(ife_path)
    if len(gsm)!=N or len(ife)!=N:
        print(stage,'incomplete:',len(gsm),'GSM8K and',len(ife),'IFEval; skip score')
        continue
    lookup={r['prompt']:r['response'] for r in ife}
    strict=[]; loose=[]
    for key,row in ife_examples:
        inp=evaluation_lib.InputExample(
            key=key, instruction_id_list=row['instruction_id_list'],
            prompt=row['prompt'],kwargs=clean_ifeval_kwargs(row)
        )
        strict.append(evaluation_lib.test_instruction_following_strict(inp,lookup))
        loose.append(evaluation_lib.test_instruction_following_loose(inp,lookup))
    strict_vals=[int(z.follow_all_instructions) for z in strict]
    loose_vals=[int(z.follow_all_instructions) for z in loose]
    summary.append({
        'stage':stage,'gsm8k_accuracy_pct':100*sum(z['correct'] for z in gsm)/N,
        'ifeval_strict_prompt_pct':100*sum(strict_vals)/N,
        'ifeval_loose_prompt_pct':100*sum(loose_vals)/N,
        'gsm8k_n':N,'ifeval_n':N,
    })
    for tag, outputs in [('strict',strict),('loose',loose)]:
        evaluation_lib.write_outputs(str(OUT/f'{stage}_ifeval_{tag}_{N}.jsonl'),outputs)
results=pd.DataFrame(summary)
print(results.to_string(index=False))
results.to_csv(OUT/f'summary_{N}.csv',index=False)
print('Saved',OUT/f'summary_{N}.csv')

ERROR:absl:Unable to detect language for text **** due to No features in text.
ERROR:absl:Unable to detect language for text **** due to No features in text.
ERROR:absl:Unable to detect language for text ****** due to No features in text.
ERROR:absl:Unable to detect language for text ****** due to No features in text.


stage  gsm8k_accuracy_pct  ifeval_strict_prompt_pct  ifeval_loose_prompt_pct  gsm8k_n  ifeval_n
 Base                 3.0                      12.5                     13.5      200       200
  SFT                60.0                      46.0                     48.5      200       200
  DPO                59.5                      64.5                     69.0      200       200
 RLVR                66.5                      60.5                     67.0      200       200
Saved /content/drive/MyDrive/P3b_OLMo_results/summary_200.csv


In [19]:
# Print SAME example IDs across four checkpoints for qualitative analysis.
for task in ['gsm8k','ifeval']:
    print('\n===',task,'===')
    records={stage:{r['id']:r for r in jsonl_read(OUT/f'{stage}_{task}_{N}.jsonl')}
             for stage in MODEL_IDS}
    for ex_id in list(records['Base'])[:3]:
        print('\nExample ID:',ex_id, 'Prompt:',records['Base'][ex_id]['prompt'][:350])
        for stage in MODEL_IDS:
            if ex_id in records[stage]:
                r=records[stage][ex_id]
                print(stage,':',r['response'][:550].replace('\n',' '),
                      ('CORRECT='+str(r['correct'])) if task=='gsm8k' else '')


=== gsm8k ===

Example ID: 0 Prompt: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Base : 16 eggs x $2 = $32 Solve this plz.  A = 2A - 3A Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A = 1 Solve this plz.  A  CORRECT=False
SFT : Janet’s ducks lay 16 eggs per day. She eats 3 eggs for breakfast and bakes muffins with 4 eggs. So, she

## Submission checklist

- Use the **N=200** output, not the N=3 smoke test.
- Report prompt-level **strict** IFEval accuracy and GSM8K numeric exact match, along with examples from the final cell.
- Four checkpoints use identical examples and greedy decoding, but the Base is necessarily plain completion while instruction-tuned stages use their own chat templates.
- There is no direct safety test in these two benchmarks. Any statements about safety, calibration, diversity must be clearly qualified as **not evaluated**.
- Generated responses/summary remain in `MyDrive/P3b_OLMo_results`; save notebook itself to Drive too.